# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [14]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [15]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv
import chromadb
import json
from pydantic import BaseModel, Field
from tavily import TavilyClient
from datetime import datetime
from typing import List
from lib.messages import BaseMessage
from chromadb.utils import embedding_functions

In [16]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [17]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game
@tool
def retrieve_game(query: str):
    """
    Semantic searchs the most relevant game entries in the local vector database.
    args:
        query (str): A question about a game or the game industry.

    Returns
        A list of the most relevant matches, each containing:
        - Platform
        - Name
        - YearOfRelease
        - Description
    """

    db_path = os.path.abspath("chromadb")
    chroma_client = chromadb.PersistentClient(path=db_path)

    embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
        api_key=os.getenv("OPENAI_API_KEY"),
        api_base=os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1"),
        model_name="text-embedding-3-small",
    )

    collection = chroma_client.get_collection(
        name="udaplay",
        embedding_function=embedding_fn
    )

    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []

    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        retrieved.append({
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
        })

    return retrieved



In [18]:
results = retrieve_game("Which Mario game was the first 3D platformer?")
print(results)

[{'Platform': 'Nintendo 64', 'Name': 'Super Mario 64', 'YearOfRelease': 1996, 'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach."}, {'Platform': 'Super Nintendo Entertainment System (SNES)', 'Name': 'Super Mario World', 'YearOfRelease': 1990, 'Description': 'A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.'}, {'Platform': 'GameCube', 'Name': 'Super Smash Bros. Melee', 'YearOfRelease': 2001, 'Description': 'A crossover fighting game featuring characters from various Nintendo franchises battling it out in dynamic arenas.'}]


#### Evaluate Retrieval Tool

In [19]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

class EvaluationResult(BaseModel):
    useful: bool = Field(
        ...,
        description="Whether the retrieved documents are sufficient to answer the question."
    )
    description: str = Field(
        ...,
        description="What the model thinks about the quality of the retrieved documents."
    )


@tool
def evaluate_retrieval(question: str, retrieved_docs: list):
    """
    Evaluates the usability of the retrieved documents to answer the user's question.
    
    Args:
        question (str): The original question from the user.
        retrieved_docs (list): A list of retrieved documents most similar to the user query in the Vector Database.

    Returns:
        dict: A dictionary containing:
            - useful (bool): Whether the documents are useful to answer the question.
            - description (str): A detailed explanation about the evaluation result.
    """

    if not question or not isinstance(question, str):
        return {
            "useful": False,
            "description": "No valid question was provided."
        }

    if not retrieved_docs:
        return {
            "useful": False,
            "description": (
                f"The query '{question}' returned no documents. "
                "There is not enough evidence to answer the question reliably."
            )
        }

    formatted_docs = []
    for i, doc in enumerate(retrieved_docs, start=1):
        if isinstance(doc, dict):
            formatted_docs.append(
                f"{i}. {doc.get('Name', 'Unknown')} | "
                f"Platform: {doc.get('Platform', 'N/A')} | "
                f"Year: {doc.get('YearOfRelease', 'N/A')} | "
                f"Description: {doc.get('Description', doc.get('content', ''))}"
            )
        else:
            formatted_docs.append(str(doc))


    prompt = f"""
    You are evaluating a retrieval system for a video game research assistant.

    User Question:
    "{question}"

    Retrieved Documents:
    {json.dumps(formatted_docs, ensure_ascii=False)}

    Return ONLY valid JSON in this exact format:
    {{
      "useful": true,
      "description": "short explanation"
    }}

    Instructions:
    - "useful" must be either true or false
    - "description" must explain clearly if the documents are enough to answer the question
    - If the documents are weak or missing facts, say that clearly
    - Do not include anything outside the JSON
    """
                                                                                        
    llm = LLM(api_key=OPENAI_API_KEY)
    response = llm.invoke(prompt, response_format=EvaluationResult)

    if hasattr(response, "parsed") and response.parsed is not None:
        evaluation_result = response.parsed
    else:
        raw = (response.content or "").strip()
        if raw.startswith("```"):
            raw = raw.strip("`")
            if raw.lower().startswith("json"):
                raw = raw[4:].strip()

        try:
            evaluation_result = EvaluationResult.model_validate_json(raw)
        except Exception:
            try:
                parsed = json.loads(raw)
                evaluation_result = EvaluationResult(**parsed)
            except Exception:
                return {
                    "useful": False,
                    "description": "The model did not return a valid evaluation result."
                }

    return {
        "useful": evaluation_result.useful,
        "description": evaluation_result.description
    }

In [20]:
retrieved = retrieve_game("When was the Spider-Man game released?")

evaluation = evaluate_retrieval(
    question="When was the Spider-Man game released?",
    retrieved_docs=retrieved
)

print(evaluation)

{'useful': True, 'description': "The retrieved documents include the release year of the Spider-Man game, specifically stating that Marvel's Spider-Man was released in 2018, which directly answers the user's question."}


#### Game Web Search Tool

In [21]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question: str):
    """
    Search the web for current information about games when the local vector DB is not enough.

    Args:
        question (str): A question about a game or the video game industry.

    Returns:
        list: A normalized list of search result dictionaries with:
            - title
            - url
            - content
    """
    if not question or not isinstance(question, str):
        return []

    try:
        client = TavilyClient(api_key=TAVILY_API_KEY)
        search_result = client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        include_raw_content=False,
        include_images=False
        )

        formatted_results = {
            "answer": search_result.get("answer", ""),
            "results": search_result.get("results", []),
            "search_metadata": {
                "timestamp": datetime.now().isoformat(),
                "query": question
            }
        }

        return formatted_results

    except Exception as e:
        return [
            {
                "answer": "Search failed",
                "results": f"Web search failed: {str(e)}",
                "search_metadata": {
                    "timestamp": datetime.now().isoformat(),
                    "query": question
                }
            }
        ]

In [22]:
results = game_web_search("When was GTA6 released?")
print(results)

{'answer': "GTA 6's release date is set for November 19, 2026, as confirmed by multiple sources including GamesRadar+, BBC News, Red Bull, and a Reddit post. Rockstar Games officially announced this date in November 2025, after previously delaying the release from May 2026 and an earlier expected Fall 2025 launch. The game will be available on PlayStation 5 and Xbox Series X and S, with a PC release date yet to be confirmed.", 'results': [{'url': 'https://www.gamesradar.com/gta-6-guide', 'title': 'GTA 6: Everything we know so far | GamesRadar+', 'content': 'The upcoming adventure in Leonida is already believed to be on track for the strongest pre-order campaign ever recorded, and as we continue to countdown to GTA 6, you can find a helpful recap of all that\'s been revealed about GTA 6 so far with a detailed recap of everything that you need to know about Rockstar\'s next game.\n\n### GTA 6 release date\n\n## GTA 6 release date\n\nGTA 6\n\nGTA 6\n\nGTA 6\'s release date is set for Nove

### Agent

In [23]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
    
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You are UdaPlay, an AI research agent for the video game industry. "
        "MANDATORY WORKFLOW: "
        "1) call retrieve_game first, "
        "2) then call evaluate_retrieval on those results, "
        "3) only if evaluate_retrieval says the documents are not useful, "
        "then call game_web_search. "
        "Do not skip the evaluation step. "
        "If the retrieved documents are useful, answer from them and do not search the web. "
        "If not useful, use web search and cite sources."
    ),
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.2
)

def answer_with_required_tool_order(question: str):
    retrieved = retrieve_game(question)
    evaluation = evaluate_retrieval(question=question, retrieved_docs=retrieved)

    if evaluation.get('useful'):
        return {
            'question': question,
            'tools_used': ['retrieve_game', 'evaluate_retrieval'],
            'retrieved_docs': retrieved,
            'evaluation': evaluation,
            'web_results': None
        }

    web_results = game_web_search(question)
    return {
        'question': question,
        'tools_used': ['retrieve_game', 'evaluate_retrieval', 'game_web_search'],
        'retrieved_docs': retrieved,
        'evaluation': evaluation,
        'web_results': web_results
    }


In [24]:
def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

run = agent.invoke("Which game was the first 3D platformer in the Mario series?")
final_state = run.get_final_state()
print_messages(final_state["messages"])


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
 -> (role = system, content = You are UdaPlay, an AI research agent for the video game industry. MANDATORY WORKFLOW: 1) call retrieve_game first, 2) then call evaluate_retrieval on those results, 3) only if evaluate_retrieval says the documents are not useful, then call game_web_search. Do not skip the evaluation step. If the retrieved documents are useful, answer from them and do not search the web. If not useful, use web search and cite sources., tool_calls = None)
 -> (role = user, content = Which game was the first 3D platformer in the Mario series?, tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessa

In [25]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?"
]

for question in questions:
    result = answer_with_required_tool_order(question)

    print(f"\nQuestion : {question}")
    print("Tools used: ", result["tools_used"])
    print("Retrieved Docs: ", result["retrieved_docs"])
    print("Evaluation: ", result["evaluation"])
    if result["web_results"]:
        print("Web Results: ", result["web_results"])
        print("Sources:")
        for item in result["web_results"].get("results", []):
            url = item.get("url")
            if url:
                print(f"- {url}")
    else:
        print("Web Results: None")


Question : When Pokémon Gold and Silver was released?
Tools used:  ['retrieve_game', 'evaluate_retrieval', 'game_web_search']
Retrieved Docs:  [{'Platform': 'Game Boy Color', 'Name': 'Pokémon Gold and Silver', 'YearOfRelease': 1999, 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.'}, {'Platform': 'Game Boy Advance', 'Name': 'Pokémon Ruby and Sapphire', 'YearOfRelease': 2002, 'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.'}, {'Platform': 'Nintendo 64', 'Name': 'Super Mario 64', 'YearOfRelease': 1996, 'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach."}]
Evaluation:  {'useful': False, 'description': "The retrieved documents do not provide the release date for Pokémon Gold and Silver, making them insufficient to answer the user's question."}
Web Results:  {'answer': 'Pokémon Gold and

### (Optional) Advanced

In [26]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes